#  MODIS Water Cluster Training

GPU or CPU enabled

Date modified: 8/12/2024 by Amanda Burke

In [ ]:
from sklearn.cluster import KMeans
from pathlib import Path   
import pandas as pd
import numpy as np
# import cupy as cp
import warnings
import pickle
import glob
import csv
import os

%run MW_ML_Functions.ipynb

## Notes

- 8/12/2024: The GPU scripts in MW_ML_Functions.ipynb cannot load the cudl, so subsequently cannot use here

## Parameters and Functions

In [ ]:
GPU = False

In [ ]:
MODEL = 'rf'
TEST_RATIO = 0.2
RANDOM_STATE = 42
LABEL_NAME = 'water'
if GPU is False:
    DATA_TYPE = np.int16
else: 
    DATA_TYPE = cp.float32
FRAC_LAND=0.5
num_datapoints = 100000000

In [ ]:
outlier_threshold = None 
# outlier_threshold = 10000

"Unhighlight" different versions for input

In [ ]:
# #############################
# # VERSION 4.2.1 (targeted 500k points)
# TILE_IN = 'Golden'#v4.2.1
# DATA_VERSION='v4.2.1'
# offsets_indexes = ['x_offset', 'y_offset', 'year', 'julian_day','tileID']
# #############################

##############################
#VERSION 2.0.1 (5 million points)
TILE_IN = 'GLOBAL'#v2.0.1
DATA_VERSION='v2.0.1'
offsets_indexes = ['x_offset', 'y_offset', 'year', 'julian_day']
##############################

# #############################
# #VERSION 0.0.0 (2billion data points)
# TILE_IN = 'cleaned'#v2.0.1
# DATA_VERSION='AGU'
# offsets_indexes = []#'x_offset', 'y_offset', 'year', 'julian_day']
# ##############################

training_data_basepath = f'/explore/nobackup/projects/ilab/data/MODIS/MODIS_WATER_ML/training_data/{DATA_VERSION}'
glob_string = os.path.join(training_data_basepath,'MOD*{}*.parquet.gzip'.format(TILE_IN))
data_paths = sorted([fv for fv in glob.glob(glob_string)])

print(data_paths)
data_path = data_paths[0]
print(data_path)

Change the input features below

In [ ]:
#Comment the different variables that are *wanted* for training 
colsToDrop = [
            # 'sur_refl_b01_1','sur_refl_b02_1',
            'sur_refl_b03_1','sur_refl_b04_1',
            'sur_refl_b05_1','sur_refl_b06_1',
            # 'sur_refl_b07_1', 'ndvi',
            'ndwi1','ndwi2'
            ]

colsToDropTraining = colsToDrop.copy()
colsToDropTraining.extend(offsets_indexes)
v_names = ['sur_refl_b01_1','sur_refl_b02_1',
           'sur_refl_b03_1','sur_refl_b04_1',
           'sur_refl_b05_1','sur_refl_b06_1',
           'sur_refl_b07_1','ndvi',
           'ndwi1','ndwi2']

In [ ]:
colsToDrop

## Input data

In [ ]:
load_data_params = {'fpath':data_path,'colsToDrop':colsToDropTraining,
                    'dataType':DATA_TYPE,'splitXY':True,
                    'imbalance':False,'trainTestSplit':True}

In [ ]:
%%time

if GPU is True: 
    X, X_test, y, y_test = GPU_Load_Data(**load_data_params)
else: 
    X, X_test, y, y_test = CPU_Load_Data(**load_data_params)
    
X = X.iloc[:num_datapoints,:] 
y = y.iloc[:num_datapoints] 

X_test = X_test.iloc[:num_datapoints,:] 
y_test = y_test.iloc[:num_datapoints] 

print(f'data shape: {X.shape}, {y.shape}')

Thresholding out outliers

In [ ]:
if outlier_threshold is not None:
    #keep the values below the outlier threshold
    X_no_outlier = X[X['sur_refl_b01_1'] < outlier_threshold]
    y_no_outlier = y.loc[X_no_outlier.index]

    print(f'Removing {len(X) - len(X_no_outlier)} outliers')

Separating land and water datapoints

In [ ]:
if outlier_threshold is None:
    features = X
    label = y
else: 
    print(f'Dataset with no outliers greater than {outlier_threshold}')
    features = X_no_outlier
    label = y_no_outlier
    
#Getting the indices that are associated with land (0) and water (1)
water_ind = np.where(label>0.5)[0]
land_ind = np.where(label<0.5)[0]

#Subset the X AND y data to later/ subset with the clusters and then combine for RFA
X_water = features.iloc[water_ind,:]
y_water = label.iloc[water_ind]

X_land = features.iloc[land_ind,:]
y_land = label.iloc[land_ind]

print(f'Water datapoints: {len(X_water)}, Land datapoints: {len(X_land)}')

In [ ]:
_ = [print(column) for column in X.columns]

## Clustering

Based on the cluster analysis above on 5.03.23, 15 clusters appears to have the most data and exclude outliers so will use that number for selection 

In [ ]:
kmean_land_fit_file = 'Non_python_files/kmeans_land_fit.pkl'
kmean_water_fit_file = 'Non_python_files/kmeans_water_fit.pkl'

kme_water = pickle.load(open(kmean_water_fit_file, 'rb'))
km_water_out = kme_water.predict(X_water)

kme_land = pickle.load(open(kmean_land_fit_file, 'rb'))
km_land_out  = kme_land.predict(X_land)

In [ ]:
# CLUSTER_NUM=15

# common_params = {
#     "random_state": 42,
#     "init":"random"
# }

# kme_water = KMeans(n_clusters=CLUSTER_NUM, **common_params).fit(X_water)
# km_water_out = kme_water.predict(X_water)

# kme_land = KMeans(n_clusters=CLUSTER_NUM, **common_params).fit(X_land)
# km_land_out = kme_land.predict(X_land)

## pickle.dump(kme_land_random, open("kmeans_land_fit.pkl", "wb"))
## pickle.dump(kme_water_random, open("kmeans_water_fit.pkl", "wb"))

### Evenly balanced cluster data

In [ ]:
land_count = Cluster_Counts(km_water_out)
water_count = Cluster_Counts(km_land_out)

#Needs the smallest cluster size between the land and water labels
SMALLEST_COUNT = np.nanmin([land_count,water_count])
print('Smallest cluster count:', SMALLEST_COUNT)

In [ ]:
cluster_water = Stratified_Cluster_Sampling(SMALLEST_COUNT,km_water_out)
cluster_land = Stratified_Cluster_Sampling(SMALLEST_COUNT,km_land_out)

#### Combining even balance cluster data

In [ ]:
X_seperate_cluster = pd.concat([
    X_land.iloc[cluster_land],X_water.iloc[cluster_water]
    ])
    
y_seperate_cluster = pd.concat([
    y_land.iloc[cluster_land],y_water.iloc[cluster_water]
    ])

#Combine the data so that we can shuffle the indices and keep the data together that should be
all_cluster = pd.concat([X_seperate_cluster,y_seperate_cluster],axis=1).sample(frac=1)
X_cluster = all_cluster[X_seperate_cluster.columns]
y_cluster = all_cluster['water']

print(X_cluster,y_cluster)

### Proportional cluster data

In [ ]:
# # List of the clusters: kmeans_output_land and kmeans_output_water
# # Data: X_water, X_land, y_water, y_land

# PERCENT_RANDOM_PULL = 0.15

In [ ]:
# # np.random.seed(42)
# random_ind_land = np.array([])
# random_ind_water = []

# for cluster in np.unique(kmeans_output_water_random):
#     print(f'cluster {cluster}')
#     cluster_ind_water = np.where(kmeans_output_water_random == cluster)[0]
#     # cluster_ind_water = np.where(bgm_water == cluster)[0]
#     COUNT_RANDOM_PULL_WATER = int(PERCENT_RANDOM_PULL*len(cluster_ind_water))
#     random_pts_water = np.random.choice(cluster_ind_water,COUNT_RANDOM_PULL_WATER,replace=False)
#     max_X_random_water = np.nanmax(X_water['sur_refl_b01_1'].iloc[random_pts_water])
#     if outlier_threshold is None: 
#         #remove the entire cluster with outlier not just single datapoints
#         if max_X_random_water > 10000:
#             print(f'contains outliers')
#             continue
#         else: 
#             random_ind_water = np.append(random_ind_water, random_pts_water)
#     cluster_ind_land = np.where(kmeans_output_land_random == cluster)[0]
#     # cluster_ind_land = np.where(bgm_land == cluster)[0]
#     COUNT_RANDOM_PULL_LAND = int(PERCENT_RANDOM_PULL*len(cluster_ind_land))
#     random_pts_land = np.random.choice(cluster_ind_land,COUNT_RANDOM_PULL_LAND,replace=False)
#     random_ind_land = np.append(random_ind_land, random_pts_land)
#     print(f'Pulling {COUNT_RANDOM_PULL_WATER} Water pts and {COUNT_RANDOM_PULL_LAND} Land pts')
#     print()
# random_ind_water = random_ind_water.astype('int')
# random_ind_land = random_ind_land.astype('int')

# print(random_ind_water,random_ind_land)

### Creating random sample, same size as clusters

In [ ]:
np.random.seed(42)

match_sample_land = np.random.choice( np.arange(len(X_land)),len(cluster_water),replace=False)
match_sample_water = np.random.choice( np.arange(len(X_water)),len(cluster_land),replace=False)

X_seperate_match= pd.concat([
    X_land.iloc[match_sample_land],X_water.iloc[match_sample_water]
        ])
y_seperate_match = pd.concat([
    y_land.iloc[match_sample_land],y_water.iloc[match_sample_water]
        ])

all_match = pd.concat([X_seperate_match,y_seperate_match],axis=1).sample(frac=1).reset_index(drop=True)
X_match= all_match[X_seperate_match.columns]
y_match = all_match['water']

print(all_match)
print(X_match)
print(y_match)

## Random forest

Tuning(obj, X_chosen, y_chosen, 
           ml_model = skRF, search_space = rf_search_space, Training = False,
           outfile = f'rfa_models/MODIS_RFA_v201_Cluster_no-outlier-cluster',
           save_pkl = False, NN = False, NTRIALS = 2):

----------------------------

*Change the tuning parameters below e.g.*

obj = CPU_RF_Objective v. GPU_RF_Objective

X_chosen = X_cluster v. X_match v. X

y_chosen = y_cluster v. y_match v. y

ml_model = 'CPU' v 'GPU'

out_file = 'rfa_models/MODIS_RFA_v201_EBcluster'
v.
f'rfa_models/MODIS_RFA_v201_EBmatch'

----------------------------
*If you want to output the saved ml model, set*

Training = True

If CPU: 

In [ ]:
Tuning(CPU_RF_Objective, X_cluster, y_cluster, NTRIALS = 2)

In [ ]:
Tuning(CPU_RF_Objective, X_match, y_match, NTRIALS = 2,
      out_file = f'rfa_models/MODIS_RFA_v201_EBmatch_MaxScore')

In [ ]:
# Tuning(CPU_RF_Objective, X, y, NTRIALS = 10,
#       out_file = f'rfa_models/MODIS_RFA_v201_MaxScore')

If GPU:

In [ ]:
# Tuning(GPU_RF_Objective, X_cluster, y_cluster, NTRIALS = 10, ml_model = 'GPU')

In [ ]:
# Tuning(GPU_RF_Objective, X_match, y_match, NTRIALS = 10, ml_model = 'GPU',
#       out_file = f'rfa_models/MODIS_RFA_v201_EBmatch_MaxScore')

In [ ]:
# Tuning(GPU_RF_Objective, X, y, NTRIALS = 10, ml_model = 'GPU',
#       out_file = f'rfa_models/MODIS_RFA_v201_MaxScore')